In [1]:
from langchain_community.chat_models import ChatTongyi
from langchain_core.tools import tool
from langchain.agents import create_agent

C:\Users\86178\AppData\Local\Temp\ipykernel_3492\3884030558.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.chat_models import ChatTongyi


In [2]:
# 2. 工具调用
# 2.1. 模拟商品数据库
product_db = {
    "iPhone": {"id": 101, "price": 5999},
    "MacBook": {"id": 102, "price": 12999},
    "小米手机": {"id": 103, "price": 3999},
}

# 2.2. 工具A：根据名称获取ID
@tool(description="根据商品名称获取对应的商品ID")
def get_product_id_by_name(name: str) -> int | None:
    """输入商品名称（如'iPhone'），返回商品ID"""
    name_lower = name.lower()
    for prod_name, info in product_db.items():
        if prod_name.lower() == name_lower:
            print(f"[工具A] 商品 '{name}' 的 ID 是 {info['id']}")
            return info['id']
    print(f"[工具A] 未找到商品 '{name}'")
    return None

# 2.3. 工具B：根据ID获取价格
@tool(description="根据商品ID查询商品价格")
def get_product_price_by_id(product_id: int) -> str | None:
    """输入商品ID，返回价格字符串"""
    for prod_name, info in product_db.items():
        if info['id'] == product_id:
            price = info['price']
            print(f"[工具B] ID {product_id} 的商品 '{prod_name}' 价格为 {price} 元")
            return f"{prod_name} 的价格是 {price} 元"
    print(f"[工具B] 未找到 ID {product_id} 对应的商品")
    return None

In [3]:
# 3. 创建 Agent
agent = create_agent(
    model=ChatTongyi(model="qwen3-max"),
    tools=[get_product_id_by_name, get_product_price_by_id],
    system_prompt="你是一个商品查询助手。当用户询问某个商品的价格时，请先调用 get_product_id_by_name 获取商品ID，"
           "然后再调用 get_product_price_by_id 获取价格，最后把价格告知用户。"
)

In [4]:
# 4. 调用 Agent 执行任务
user_query = "我想知道小米手机的价格是多少？"
print(f"用户问题：{user_query}\n")

# 调用 agent 并打印结果
result = agent.invoke({"messages": [
    {"role": "user", "content": user_query}
]})

用户问题：我想知道小米手机的价格是多少？

[工具A] 商品 '小米手机' 的 ID 是 103
[工具B] ID 103 的商品 '小米手机' 价格为 3999 元


In [5]:
# 5. 提取最终回答
# final_answer = result["messages"][-1].content
print(f"\n最终回答：{result}")


最终回答：{'messages': [HumanMessage(content='我想知道小米手机的价格是多少？', additional_kwargs={}, response_metadata={}, id='6620ad3c-3caa-456d-92f8-991cfa725d83'), AIMessage(content='', additional_kwargs={'tool_calls': [{'function': {'arguments': '{"name": "小米手机"}', 'name': 'get_product_id_by_name'}, 'id': 'call_f8e6c6dd16e547db9b905380', 'index': 0, 'type': 'function'}]}, response_metadata={'model_name': 'qwen3-max', 'finish_reason': 'tool_calls', 'request_id': 'f416c870-b10a-983b-8f43-8e471fd7028c', 'token_usage': {'input_tokens': 368, 'output_tokens': 24, 'prompt_tokens_details': {'cached_tokens': 0}, 'total_tokens': 392}}, id='lc_run--019eca5d-1b59-7673-b38c-1a41e2f61964-0', tool_calls=[{'name': 'get_product_id_by_name', 'args': {'name': '小米手机'}, 'id': 'call_f8e6c6dd16e547db9b905380', 'type': 'tool_call'}], invalid_tool_calls=[]), ToolMessage(content='103', name='get_product_id_by_name', id='9b7d2591-0b10-4dd7-b43b-6af7a9c691b4', tool_call_id='call_f8e6c6dd16e547db9b905380'), AIMessage(content='',